In [10]:
import numpy as np
import pandas as pd
import pandas_ta as ta
from Engine.DataTimeseries import DataTimeseries
from Engine.Backtester import Backtester

In [11]:
data = DataTimeseries('./Data/nq_continous.csv', symbol='NQ')
data.resample_data('5min')
close_series: pd.Series = data.get_close_prices_series()
close_series


Loading ./Data/nq_continous.csv

Start:     2010-06-07 00:00:00+00:00
End:       2025-11-23 23:59:00+00:00
Timezone:  UTC
Timeframe: 1min
Loaded 5,224,896 clean rows



Time
2010-06-07 00:05:00+00:00     1827.25
2010-06-07 00:10:00+00:00     1826.75
2010-06-07 00:15:00+00:00     1821.75
2010-06-07 00:20:00+00:00     1820.50
2010-06-07 00:25:00+00:00     1819.25
                               ...   
2025-11-23 23:40:00+00:00    24475.00
2025-11-23 23:45:00+00:00    24478.00
2025-11-23 23:50:00+00:00    24490.25
2025-11-23 23:55:00+00:00    24472.00
2025-11-24 00:00:00+00:00    24471.50
Length: 1084692, dtype: float64

In [12]:
EMSD_SHORT_WEIGHT = 1
EMSD_LONG_WEIGHT = 0
EMSD_SHORT_LENGTH = 64
EMSD_LONG_LENGTH = 256

In [13]:
close = close_series

# N and 4N pairs
ema_8 = ta.ema(close, 8)
ema_32 = ta.ema(close, 32)

ema_16 = ta.ema(close, 16)
ema_64 = ta.ema(close, 64)

ema_256 = ta.ema(close, 256)
# print(close.ewm(span=EMSD_SHORT_LENGTH).std())
# emsd = close.ewm(span=EMSD_SHORT_LENGTH).std() * EMSD_SHORT_WEIGHT + close.ewm(span=EMSD_LONG_LENGTH).std() * EMSD_LONG_WEIGHT

# drop all nan values and sync them up
sync_df = pd.DataFrame({
    'ema8': ema_8,
    'ema16': ema_16,
    'ema32': ema_32,
    'ema64': ema_64,
    'ema256': ema_256,
}).dropna()

ema_8 = sync_df['ema8']
ema_16 = sync_df['ema16']
ema_32 = sync_df['ema32']
ema_64 = sync_df['ema64']
ema_256 = sync_df['ema256'] 


# calculate forecasts
signal_8_32 = (ema_8 - ema_32)
signal_16_64 = (ema_16 - ema_64)
signal_64_256 = (ema_64 - ema_256)

volitility_8_32 = signal_8_32.ewm(span=EMSD_SHORT_LENGTH).std() * EMSD_SHORT_WEIGHT + signal_8_32.ewm(span=EMSD_LONG_LENGTH).std() * EMSD_LONG_WEIGHT
volitility_16_64 = signal_16_64.ewm(span=EMSD_SHORT_LENGTH).std() * EMSD_SHORT_WEIGHT + signal_16_64.ewm(span=EMSD_LONG_LENGTH).std() * EMSD_LONG_WEIGHT
volitility_64_256 = signal_64_256.ewm(span=EMSD_SHORT_LENGTH).std() * EMSD_SHORT_WEIGHT + signal_64_256.ewm(span=EMSD_LONG_LENGTH).std() * EMSD_LONG_WEIGHT

raw_forecast_8_32 = signal_8_32 / volitility_8_32
raw_forecast_16_64 = signal_16_64 / volitility_16_64
raw_forecast_64_256 = signal_64_256 / volitility_64_256

df = pd.DataFrame({
    '8_32': raw_forecast_8_32,
    '16_64': raw_forecast_16_64,
    '64_256': raw_forecast_64_256,
    '8_ema': ema_8,
    '16_ema': ema_16,
    '32_ema': ema_32,
    '64_ema': ema_64,
    '256_ema': ema_256,
    'close': close
})



In [ ]:
import optuna

weight_start = -30
weight_end = 30
no_trade_range = 20
np_8_32 = df['8_32'].to_numpy(dtype=np.float32)
np_16_64 = df['16_64'].to_numpy(dtype=np.float32)
np_64_256 = df['64_256'].to_numpy(dtype=np.float32)

clip_min = -20
clip_max = 20
np_8_32 = np.clip(np_8_32, clip_min, clip_max)
np_16_64 = np.clip(np_16_64, clip_min, clip_max)
np_64_256 = np.clip(np_64_256, clip_min, clip_max)

trials = 1000

w1 = np.arange(-20, 20, 0.5)
w2 = np.arange(-20, 20, 0.5)
w3 = np.arange(-20, 20, 0.5)
w4 = np.arange(-20, 20, 0.5)

def generate_signal(w1, w2, w3, w4):
    combined_forecast = np_8_32 * w1 + np_16_64 * w2 + np_64_256 * w3
    combined_forecast = combined_forecast / clip_max
    signal = np.zeros_like(combined_forecast, dtype=np.float32)

    # long_pos = combined_forecast > w4
    # short_pos = combined_forecast < -w4
    signal[combined_forecast > w4] = 1
    signal[combined_forecast < -w4] = -1

    # signal[long_pos] = combined_forecast[long_pos]
    # signal[short_pos] = combined_forecast[short_pos]
    # # signal[d] = combined_forecast
    # signal[combined_forecast < -w4] = combined_forecast
    return pd.Series(signal, index=df.index)

def objective(trial):
    w1 = trial.suggest_float('w1', weight_start, weight_end)
    w2 = trial.suggest_float('w2', weight_start, weight_end)
    w3 = trial.suggest_float('w3', weight_start, weight_end)
    w4 = trial.suggest_float('w4', 0, no_trade_range)

    bt = Backtester(close_series, data.indicies, generate_signal(w1, w2, w3, w4).values, data.get_timeframe(), tick_value=0.25, contract_multiplier=2, initial_equity=100000, commission=0, slippage_ticks=0)
    bt.quick_backtest()

    return -bt.total_pnl 

study = optuna.create_study()
study.optimize(objective, n_trials=trials)
print(study.best_params)
print(-study.best_value)


[I 2025-11-27 18:31:40,820] A new study created in memory with name: no-name-78267925-c0a1-4674-912e-f6517d40a585
/Users/newuser/Desktop/Random Coding Crap/Futures/RugbeatunosBacktestingSuite/Engine/Backtester.py:279: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return self.close_prices[-1] / self.close_prices[0] - 1
[I 2025-11-27 18:31:40,844] Trial 0 finished with value: -99997.0 and parameters: {'w1': 0.6079723905848837, 'w2': 7.002478062470793, 'w3': 1.8398202876110545, 'w4': 6.425344764724603}. Best is trial 0 with value: -99997.0.
[I 2025-11-27 18:31:40,859] Trial 1 finished with value: -98211.5 and parameters: {'w1': -5.158975193259963, 'w2': 11.024853024600574, 'w3': -26.546777858798535, 'w4': 10.800103718306799}. Best is trial 0 with value: -99997.0.
[I 2025-11-27 18:31:40,873] Trial 2 finis

{'w1': 24.427201296061405, 'w2': 25.536695386479572, 'w3': -4.981421683267197, 'w4': 1.073011321359659}
144292.0
